# Description

```markdown
Problem      : Classification : Binary
Resolve      : Learning : Deep Learning
Field        : Natural Language Processing (NLP)
Algorithm    : Neural Network
Architecture : LSTM
Network      : Siamese
Action       : Swish relu
Loss         : Contrastive
Train        : Supervised
Input        : Feature : image pairs | Label
Output       : Binary
Dataset      : aclImdb
```

# General

Import

In [ ]:
import os
import numpy as np
import cv2 as cv
from matplotlib import pyplot as plt
import keras as ks
import tensorflow as tf

Variable

In [ ]:
ROOT_DIR = '/Volumes/data/documents/ai_document'
DATA_DIR = '/Volumes/data/develop/ai'
PATH_DATASET_TRAIN = os.path.sep.join([DATA_DIR, "dataset", "aclImdb", "train"])
PATH_DATASET_TEST = os.path.sep.join([DATA_DIR, "dataset", "aclImdb", "test"])
EPOCHS = 64
BATCH_SIZE = 32
MAX_LENGHTS = 600
MAX_TOKENS = 20000

# Dataset

Train

In [ ]:
train_ds = ks.utils.text_dataset_from_directory(
    PATH_DATASET_TRAIN, batch_size=BATCH_SIZE
)

Test

In [ ]:
test_ds = ks.utils.text_dataset_from_directory(
    PATH_DATASET_TEST, batch_size=BATCH_SIZE
)

Shape

In [ ]:
for inputs, targets in train_ds:
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)
    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)
    print("inputs[0]:", inputs[0])
    print("targets[0]:", targets[0])
    break

In [ ]:
text_only_train_ds = train_ds.map(lambda x, y: x)

text_vectorization = ks.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=MAX_LENGHTS,
)

text_vectorization.adapt(text_only_train_ds)


int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

# Model

contrastive_loss

In [ ]:
inputs = ks.Input(shape=(None,), dtype="int64")
embedded = ks.layers.Embedding(input_dim=MAX_TOKENS, output_dim=256)(inputs)
x = ks.layers.LSTM(32)(embedded)
outputs = ks.layers.Dense(1, activation="sigmoid")(x)
model = ks.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])
              
model.summary()

# Train

In [ ]:
model.fit(int_train_ds, validation_data=int_test_ds, epochs=10)

# Evaluation

### Loss and Accuracy

### History

loss

accuracy

# Save model

# Test